# Winrates with vs without my friend
[← back to readme](../readme.md)

I spent the majority of the sample playing with one of my friends.
He has significantly higher elo than I do.
I will hereby compare my winrate when I play with him versus when I play without him.

The real question I am asking here is how well can League of Legends matchmaking handle large elo gaps between premade players in ranked flex and draft games (we usually play those two game modes together).

I am using a permutation test with significance threshold of 5 / 3 / 2 % = 0,83 %.
The 3 is a part of Bonferroni's correction. The 2 is there, since I am performing a two-sided permutation test, since matchmaking could be overcompensating and thus crushing my winrate with him.


## Imports

In [2]:
import csv

from permutation_distribution import *

## Load the aggregated data

In [3]:
rows = []

with open("friend_data.csv", encoding="utf-8") as f:
  reader = csv.reader(f)

  next(reader)  # skip header

  for win, with_friend in reader:
    rows.append((win == "True", with_friend == "True"))

## Compare win rates

In [4]:
sample_without_friend = [win for win, with_friend in rows if not with_friend]
without_friend_wr = sum(sample_without_friend) / len(sample_without_friend)
print(f"{without_friend_wr=:.2f}")
print(f"{len(sample_without_friend)=}")

print()

sample_with_friend = [win for win, with_friend in rows if with_friend]
with_friend_wr = sum(sample_with_friend) / len(sample_with_friend)
print(f"{with_friend_wr=:.2f}")
print(f"{len(sample_with_friend)=}")
print()

without_friend_wr=0.34
len(sample_without_friend)=38

with_friend_wr=0.64
len(sample_with_friend)=74



## Permutation test

In [5]:
def wr_diff(with_friend_sample, without_friend_sample):
  with_friend_wr = sum(with_friend_sample) / len(with_friend_sample)
  without_friend_wr = sum(without_friend_sample) / len(without_friend_sample)

  return with_friend_wr - without_friend_wr


mixed_sample = [row[0] for row in rows]

dist = permutation_distribution(
  mixed_sample,
  wr_diff,
  len(sample_with_friend)
)

bottom_significant_value, top_significant_value = significant_value(
  dist,
  top_percentile=100 - (5 / 3 / 2),
  bottom_percentile=5 / 3 / 2
)

real_wr_diff = with_friend_wr - without_friend_wr

print(f"{bottom_significant_value=:.2f} {top_significant_value=:.2f}")
print(f"{real_wr_diff=:.2f}")

bottom_significant_value=-0.22 top_significant_value=0.25
real_wr_diff=0.29


The observed win rate difference of 29 % is larger than the significant value of 25 %.
While somewhat close to the border, the standard I have set for myself is very stringent and I would thus reject the null hypothesis.

I underestimated how large the values needed to prove statistical significance would be. I thought 100 samples would be easily enough. I got lucky that the difference turned out this massive.

## Commentary
While my friend is significantly better than me, I did not expect the difference to be this massive. The result suggests that ranked flex and draft queues are terrible at handling large elo discrepancies ~ they cannot deduce that my friend is doing the heavy lifting, not me.

Though alternative explanations may arise as well. Since we usually play in the bot lane together, it could be that we really are significantly better when playing together because we're used to each other's playstyle and can communicate well after all the time we've spent together.
Though I would somewhat doubt that given that I tend to prefer playing with almost any other premade partner more because of his very aggressive playstyle compared to my passive "we scale" attitude.

This finding also suggests that I should probably be controlling for this variable in order to avoid Simpson's paradox. However, due to the small expected scale of the project, I have decided not to.